# 02 – Enrichissement NLP sur Colab (GPU)

Lance les étapes lourdes du pipeline sur un GPU Colab : nettoyage (`prepare-catalog`), puis zero-shot NLI et KeyBERT (`enrich-catalog`). Le notebook ne contient aucune logique : il appelle les commandes du projet.

**Avant de commencer :** *Exécution › Modifier le type d'exécution › GPU (T4)*.

Les données et le cache sont écrits sur votre Google Drive : si la session est coupée, relancez simplement les cellules, le calcul reprend là où il s'est arrêté.

## 1. Drive et dossier de données

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/book-recommender/data"
os.makedirs(DATA_DIR, exist_ok=True)
os.environ["BOOKREC_DATA_DIR"] = DATA_DIR
os.environ["BOOKREC_DEVICE"] = "cuda"

## 2. Code du projet

In [ ]:
REPO = "https://github.com/Oumar26/book-recommender-nlp.git"
BRANCH = "master"

![ -d /content/book-recommender-nlp ] || git clone -q -b $BRANCH $REPO /content/book-recommender-nlp
!git -C /content/book-recommender-nlp pull -q
!git -C /content/book-recommender-nlp log --oneline -1
!pip install -q -e "/content/book-recommender-nlp/backend[nlp]"

In [ ]:
import torch

assert torch.cuda.is_available(), "Pas de GPU : Exécution › Modifier le type d'exécution › GPU"
print(torch.cuda.get_device_name(0))

## 3. Données et nettoyage (≈ 1 min)

In [ ]:
!book-recommender download
!book-recommender prepare-catalog

## 4. Essai sur 200 livres

Vérifie que tout fonctionne et donne une mesure de vitesse avant le passage complet. Le résultat va dans `books_sample.parquet` ; les scores calculés restent en cache et seront réutilisés.

In [ ]:
!book-recommender enrich-catalog --limit 200 --batch-size 100

## 5. Passage complet

Environ 27 500 livres × 42 hypothèses NLI, puis KeyBERT. Si la session est coupée, relancez les cellules 1, 2 puis celle-ci : le cache sur Drive évite de tout recalculer.

In [ ]:
!book-recommender enrich-catalog

## 6. Contrôle rapide

In [ ]:
import pandas as pd

books = pd.read_parquet(f"{DATA_DIR}/processed/books.parquet")
print(len(books), "livres")
books[["title", "themes", "atmospheres", "narrative_styles", "period", "keywords"]].sample(
    10, random_state=1
)

## 7. Récupérer le résultat

Téléchargez `books.parquet` depuis `MyDrive/book-recommender/data/processed/` (ou avec la cellule ci-dessous) et placez-le dans `backend/data/processed/` du projet local. Le dossier `cache/` peut rester sur Drive : il sert à recalibrer les seuils sans relancer le GPU.

In [ ]:
from google.colab import files

files.download(f"{DATA_DIR}/processed/books.parquet")